# 教程 2：写自己的训练头 —— 从一行 loss 到 CNN

教程 1 里我们是「旁观者」：把 hidden states 导出来、喂给 SAE，模型本身一根
毫毛都没动。这一篇要往前迈一大步——**让模型在生成的同时训练一个小头，并
立刻用小头的预测影响接下来的采样**。

具体做什么？每生成一个 token，就拿「第 0 层 hidden → 最后一层 hidden」
这对蒸馏信号，现场训练一个接在模型上的小回归头；头的预测经原采样器投影进
logits，引导下一个 token。这正是 ESamp 类方法的骨架。

听起来要做很多事：注册参数、反向传播、更新权重、多请求隔离……实际上你要写
的只有一个类和几个方法。如果你写过 `nn.Module`，你已经会了八成；剩下两成
是 PLLM 的「契约思维」——你不直接碰引擎的张量，而是声明你要什么、什么
时候要、结果去哪，框架核对签名后把装订好的输入交给你，梯度与参数发布由
运行时完成。

我们分三步走，每步只加一个新东西：

1. 最小的可训练 workload：一个线性头 + 一行 loss；
2. 看见训练：从 trace 里把每一步的损失捞出来；
3. 把头换成 CNN：骨架一行不动。

整篇真正的新概念只有一个：**带优化器的状态**。读懂它，后面三篇（请求级
LoRA、多轮循环、decode 训练）都是它的排列组合。

## 第一步：最小的可训练 workload

先跑完整的代码，再逐块拆解。运行之前说说该期待什么：两条 prompt 各生成
8 个 token，最后一行诊断里 `losses` 与 `updates` 计数相同、`fallback`
为 0——意思是每个训练步都真的产生了参数更新，没有一步掉进签名之外的
回退路径。

这个类里藏着四个关键设计，我们挨个看。

**先看状态。** `head = pllm.RequestState(...)` 声明了一块**请求状态**：
每条请求拿到自己独立的一份 `w`（`(H, H)` 矩阵），由 AdamW 负责更新。
你只声明布局、dtype 和优化器；版本化、按请求隔离、逐步发布全是运行时的
事。`keys=...` 决定「什么情况算同一条请求」——同一 prompt 的同一次采样
共享一条 lineage。

**再看训练。** `loss` 方法上的 `@pllm.train(state="head", ...)` 是整篇的
核心：它在说「这个函数的返回值是损失，梯度流向 `head` 的字段」。注意到
没有——你不写 `backward()`，也不写 `optimizer.step()`。返回一个标量
（或每条 lineage 一个标量），剩下的按签名完成。

**然后是数据从哪来。** `Annotated[torch.Tensor, pllm.Hidden(0)]` 取第 0 层
hidden，`Hidden(-1)` 取最后一层；`groups`（`StateIndex`）告诉你每行属于
哪条 lineage，所以损失按 lineage 归并——两个请求同批训练，梯度互不串味。

**最后是「训练得有个用处」。** `predict` + `@pllm.guide_sampling(beta=0.2)`
把头的预测经原采样器投影进 logits，这是这个 workload 的**效果**；没有效果
的纯训练不构成后端附件。而 `ready` 门控解决一个尴尬问题：第一步时 `w`
还是零初始化，拿它引导采样等于注入噪声——所以「还没训练好别说话」，
`versions > 0` 才放行。

In [1]:
from typing import Annotated
import torch
import pllm
from vllm import SamplingParams
from pllm.config import ModelConfig


class LinearDistiller(pllm.Workload):
    """每步从第 0 层 hidden 学一个线性映射到最后一层 hidden。"""

    head = pllm.RequestState(
        {"w": lambda m: (m.hidden_size, m.hidden_size)},
        dtype=torch.float32,
        optimizer=pllm.AdamW(lr=3e-3),
        keys=(pllm.state.RequestKeyField.PROMPT_IDX, pllm.state.RequestKeyField.SAMPLE_IDX),
    )

    def ready(self, versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return versions > 0     # 首个版本发布前不引导，避免零初始化参数注入噪声

    @pllm.on(pllm.Phase.BEFORE_SAMPLE, rows="ready")
    @pllm.guide_sampling(beta=0.2)
    def predict(self, source: Annotated[torch.Tensor, pllm.Hidden(0)],
                w: Annotated[torch.Tensor, pllm.Parameter("head", "w")],
                versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return torch.bmm(source.float().unsqueeze(1), w).squeeze(1)

    @pllm.on(pllm.Phase.AFTER_SAMPLE)
    @pllm.train(state="head", reduction="state_mean")
    def loss(self, source: Annotated[torch.Tensor, pllm.Hidden(0)],
             target: Annotated[torch.Tensor, pllm.Hidden(-1)],
             w: Annotated[torch.Tensor, pllm.Parameter("head", "w")]) -> torch.Tensor:
        prediction = torch.bmm(source.float().unsqueeze(1), w).squeeze(1)
        losses = (prediction - target.float()).square().mean(1)   # 每行一个损失
        return losses  # 每行一个损失，pllm 按状态求均值


llm = pllm.make_llm(workload=LinearDistiller(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", "bfloat16", 64, 2, 128, 0.2, seed=7))
outputs = llm.generate(["The sky is", "Water freezes at"],
    SamplingParams(max_tokens=8, temperature=0.0, ignore_eos=True), use_tqdm=False)
d = llm.worker_state().diagnostics
print("losses:", d.loss_count, "updates:", d.update_count, "fallback:", d.fallback_count)
for output in outputs:
    print(output.outputs[0].text)

/workspace/pllm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-09-10 18:41:15,219	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 09-10 18:41:15 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 18:41:16 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 18:41:18 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 18:41:18 [model.py:1883] Using max model len 64


... (27 outputs left out)

INFO 09-10 18:41:25 [core.py:355] init engine (profile, create kv cache, warmup model) took 1.66 s


INFO 09-10 18:41:28 [hf.py:540] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


WARNING 09-10 18:41:29 [jit_monitor.py:135] Triton kernel JIT compilation during inference: _bmm_outer_product_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


losses: 14 updates: 14 fallback: 0
 a beautiful place. It is a place
 0 degrees Celsius and boils at 


先看输出。诊断行告诉你训练回路是闭合的：有损失、有更新、零回退。生成的
文本看起来平平无奇——这正常，8 个 token 里一个随机初始化的头刚起步，
引导强度 `beta=0.2` 又刻意温和；这一篇的目标是跑通「边生成边训练」的
回路，不是让文本大变样。

运行到这里，两个疑问可能已经在脑子里了，当场回答。

**「梯度在哪？引擎端怎么跑我的类？」** 默认配置下负载与引擎同进程，你的
实例被直接使用；而反向与更新不需要你写——`@pllm.train` 声明之后，框架
按签名对返回值求梯度、走 AdamW、再把新版本发布出去。生产部署
（`ModelConfig(multiprocessing=True)`）走 EngineCore 子进程时，框架**按
引用**传输：驱动端只发送类坐标（模块 + 限定名）和字段值，worker 重新
import、重建实例、重新冷绑定，且安装时两边的编译签名必须逐字节相等，
不等就拒绝。所以「两个进程里跑的是不是同一份数学」是校验问题，不是信任
问题。

**「所以为什么范例里的 workload 都是冻结 dataclass？」** 正是为了让按引用
传输成立：字段可封闭传输、实例可用字段值在 worker 端重建。notebook 里为
了少一层仪式，我们直接写了普通类；落到 `.py` 模块、要上生产时，照
[`examples/huber.py`](../../examples/huber.py) 的样子加
`@dataclass(frozen=True)` 即可。细节见
[写一个负载](../guides/write-a-workload.md) 的生产部署小节。

## 第二步：看见训练 —— 逐步损失

诊断计数能告诉你「训练在发生」，但回答不了「训练有没有用」。好在训练任务
的损失在选择器开启时会进 trace：打开
`TraceSelection(tasks=..., tensors=True)`，每个训练步都会留下一条 `loss`
事件。task id 从签名计划里按符号名查——下面那段查找代码看一眼即可，真正
要学的是后面两行 `observe`：先开选择器跑一条新 prompt，跑完立刻关掉。

该期待什么？逐步损失应该**明显下降**——一个随机初始化的小头，几个 token
之内就能看到学习在发生。

In [2]:
from pllm.execution.trace import TraceSelection

plan = llm.attachment.plan
index = next(i for i, task in enumerate(plan.semantic_graph.tasks)
             if str(task.symbol) == "loss")
loss_id = next(task.task_id for task in plan.schedule_physical.tasks
               if task.task_id == index)

llm.observe(TraceSelection(tasks=(loss_id,), tensors=True, max_tensor_bytes=32768))
llm.generate(["The sea is"], SamplingParams(max_tokens=6, temperature=0.0,
    ignore_eos=True), use_tqdm=False)
llm.observe(TraceSelection())   # 关掉选择器

for event in llm.trace_snapshot().events:
    if event.tensor_name == "loss":
        print(f"step {event.step_id}: loss={event.tensor().item():.4f}")

step 9: loss=6.1901
step 10: loss=6.1071
step 11: loss=5.6148
step 12: loss=4.8965
step 13: loss=4.6399


每行一个训练步：损失从 6 出头一路降到 5 以下。注意这不是离线训练的 loss
曲线，而是**生成途中的在线学习**——头一边被训练，一边（`ready` 门控放行
之后）影响着自己正在生成的这条序列。几步之内降得这么快，是因为「从第 0 层
预测最后一层」里有大量容易拿到的线性结构；别指望它一直这么顺。

别忘了最后那行 `llm.observe(TraceSelection())` 是在**关**选择器——trace
是有预算的调试工具，不是常驻遥测。

## 第三步：把头换成 CNN

线性头跑通了，很自然的问题是：如果我的头不是线性层呢？比如——沿特征轴的
一维 CNN？

答案令人舒适：**骨架一行不动**。状态声明的位置、`predict`/`loss` 的时机、
`ready` 门控、按 `groups` 归并损失，全部原样保留；要改的只有字段的形状和
`forward` 里的数学。这正是契约思维的回报——时机、隔离、发布、回收都写在
签名里，你换的只是签名下面的数学。

唯一值得多看一眼的细节是**参数按行就位**：`conv`/`mix`/`bias` 读到手时
首轴就是打包行（每行取自己 lineage 的版本），所以前向用逐行广播，而不是
`torch.conv1d` 那种全局共享卷积核。记住「行永远是第一根轴」，剩下的就是
普通 PyTorch。

下面的单元格会先关掉上一个引擎、释放显存，再用 `CNNDistiller` 重建。跑完
看诊断：除了 `losses`/`updates` 照常、零回退之外，最后那条断言
`active_lineages == 0` 在说——请求结束，两条 lineage 的状态被精确回收，
没有泄漏。

In [3]:
CHANNELS = 16
KERNEL = 5


class CNNDistiller(pllm.Workload):
    """与 LinearDistiller 同骨架；头是深度可分离 CNN + 通道混合。"""

    head = pllm.RequestState(
        {"conv": (CHANNELS, KERNEL),
         "mix": (CHANNELS,),
         "bias": lambda m: (m.hidden_size,)},
        dtype=torch.float32,
        optimizer=pllm.AdamW(lr=3e-3),
        keys=(pllm.state.RequestKeyField.PROMPT_IDX, pllm.state.RequestKeyField.SAMPLE_IDX),
    )

    def forward(self, source, conv, mix, bias):
        pad = KERNEL // 2
        windows = torch.nn.functional.pad(source.float(), (pad, pad)).unfold(-1, KERNEL, 1)
        h = torch.nn.functional.silu((windows.unsqueeze(1) * conv.unsqueeze(2)).sum(-1))
        return source.float() + (h * mix.unsqueeze(-1)).sum(1) + bias

    def ready(self, versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return versions > 0

    @pllm.on(pllm.Phase.BEFORE_SAMPLE, rows="ready")
    @pllm.guide_sampling(beta=0.2)
    def predict(self, source: Annotated[torch.Tensor, pllm.Hidden(0)],
                conv: Annotated[torch.Tensor, pllm.Parameter("head", "conv")],
                mix: Annotated[torch.Tensor, pllm.Parameter("head", "mix")],
                bias: Annotated[torch.Tensor, pllm.Parameter("head", "bias")],
                versions: Annotated[torch.Tensor, pllm.StateVersion("head")]) -> torch.Tensor:
        return self.forward(source, conv, mix, bias)

    @pllm.on(pllm.Phase.AFTER_SAMPLE)
    @pllm.train(state="head", reduction="state_mean")
    def loss(self, source: Annotated[torch.Tensor, pllm.Hidden(0)],
             target: Annotated[torch.Tensor, pllm.Hidden(-1)],
             conv: Annotated[torch.Tensor, pllm.Parameter("head", "conv")],
             mix: Annotated[torch.Tensor, pllm.Parameter("head", "mix")],
             bias: Annotated[torch.Tensor, pllm.Parameter("head", "bias")]) -> torch.Tensor:
        losses = (self.forward(source, conv, mix, bias) - target.float()).square().mean(1)
        return losses  # 每行一个损失，pllm 按状态求均值

In [4]:
import gc
llm.close()
del llm
gc.collect()
torch.cuda.empty_cache()

llm = pllm.make_llm(workload=CNNDistiller(), config=ModelConfig(
    "Qwen/Qwen2.5-0.5B-Instruct", "bfloat16", 64, 2, 128, 0.2, seed=7))
try:
    outputs = llm.generate(["The sky is", "Water freezes at"],
        SamplingParams(max_tokens=8, temperature=0.0, ignore_eos=True), use_tqdm=False)
    for output in outputs:
        print(output.outputs[0].text)
    d = llm.worker_state().diagnostics
    print("losses:", d.loss_count, "updates:", d.update_count, "fallback:", d.fallback_count)
    assert d.loss_count > 0 and d.update_count > 0 and d.fallback_count == 0
    assert d.active_lineages == d.pending_state_releases == 0   # 请求结束，状态干净回收
finally:
    llm.close()

INFO 09-10 18:41:29 [api_utils.py:273] non-default args: {'dtype': 'bfloat16', 'seed': 7, 'max_model_len': 64, 'distributed_executor_backend': 'uni', 'enable_prefix_caching': False, 'gpu_memory_utilization': 0.2, 'max_num_batched_tokens': 128, 'max_num_seqs': 2, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': False, 'worker_cls': 'pllm.worker.GPUWorker', 'model': 'Qwen/Qwen2.5-0.5B-Instruct'}


WARNING 09-10 18:41:30 [arg_utils.py:1678] The global random seed is set to 7. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.


INFO 09-10 18:41:31 [model.py:645] Resolved architecture: Qwen2ForCausalLM


INFO 09-10 18:41:31 [model.py:1883] Using max model len 64


INFO 09-10 18:41:31 [kernel.py:306] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['vllm_c', 'native'], fused_add_rms_norm=['vllm_c', 'native'])


INFO 09-10 18:41:32 [core.py:121] Initializing a V1 LLM engine (v0.27.1) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=64, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=True, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=None, collec

... (9 outputs left out)

INFO 09-10 18:41:34 [kv_cache_utils.py:2236] Maximum concurrency for 64 tokens per request: 5073.25x


INFO 09-10 18:41:34 [gpu_worker.py:789] Free memory on device (11.17/23.48 GiB) on startup. Desired GPU memory utilization is (0.2, 4.7 GiB). Actual usage is 0.98 GiB for consumed memory (weights + non-torch), 0.0 GiB for peak activation, and 0.0 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=3832543232` (3.57 GiB) to fit into requested memory, or `--kv-cache-memory=10777917440` (10.04 GiB) to fully utilize gpu memory. Current kv cache memory in use is 3.72 GiB.


INFO 09-10 18:41:34 [core.py:355] init engine (profile, create kv cache, warmup model) took 0.89 s


 a beautiful place. It is a place
 0 degrees Celsius and boils at 
losses: 14 updates: 14 fallback: 0


## 小结

这一篇你写出了第一个**会改变模型行为**的 workload。回头看你真正写的只有
三样：一块带优化器的 `RequestState`、一个返回损失的 `@pllm.train` 函数、
一个带效果的 `predict`。梯度、更新、版本发布、按请求隔离、请求结束后的
精确回收——都是框架按签名替你完成的。

带走三句话：

- 训练 = 带优化器的状态 + 一个返回损失的函数；你只写前向，反向与发布归框架。
- 损失按行算、按 `StateIndex` 归并到 lineage，多请求同批互不串味。
- `ready` + `StateVersion` 是「还没训练好别生效」的标准门控；换架构只动
  字段与 `forward`，时机、隔离、发布、回收不变。

**下一步**：这一篇里每条请求有自己的一份 `w`，但所有请求用的仍是同一套
数学。下一篇 [教程 3：按请求路由的 LoRA](03_request_lora.ipynb) 把请求级
状态推到更实用的形态——每个请求一份自己的 LoRA。